# 📊 Week 2 Lab: From Simulation to Experimental Analysis
## Multi-Trial EMG & Kinematic Data with Pandas and Seaborn

**Course:** Machine Learning for Neuroscience
**Duration:** ~90 minutes
**Tools:** Python 3.x, NumPy, Pandas, Matplotlib, Seaborn

---

## Overview

In Week 1, you simulated a single reaching movement using NumPy. Now you will:
1. Run the simulation as a **batch experiment** (10 subjects × 30 trials × 3 conditions)
2. Organize results into **Pandas DataFrames**
3. Compute summary statistics with **GroupBy**
4. Create publication-quality figures with **Seaborn**
5. Compare **healthy vs. impaired** subject groups

## Bloom’s Taxonomy Roadmap

| Level | Section | What You’ll Do |
|-------|---------|----------------|
| 🟢 **Remember** | Part 1 | Build DataFrames from simulation output |
| 🟡 **Understand** | Part 2 | GroupBy, aggregation, and pivot tables |
| 🟠 **Apply** | Part 3 | Seaborn: violins, pair plots, heatmaps |
| 🟦 **Analyze** | Part 4 | Trial-averaged kinematics and FacetGrid |
| 🟣 **Evaluate** | Part 5 | Healthy vs. impaired comparison |
| 🔴 **Create** | Part 6 | Publication-quality multi-panel figure |

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.05)
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'lines.linewidth': 1.5})
np.random.seed(42)
print("Setup complete!")
print(f"  NumPy {np.__version__}, Pandas {pd.__version__}")

## Simulation Core (from Week 1)
Run this cell to load the Week 1 functions.

In [ ]:
# Week 1 Simulation Core (compact)
dt = 0.001; T = 1.5; time_vec = np.arange(0, T, dt)
L = 0.35; m_arm = 1.5; B_damp = 0.5; I_inertia = (1/3)*m_arm*L**2
theta_A = np.deg2rad(45); theta_B = np.deg2rad(90); GAIN = 6.0

def emg_burst(t, amp, cen, wid):
    return amp * np.exp(-((t - cen)**2) / (2 * wid**2))

def gen_envelopes(t, p):
    ag = emg_burst(t, p['ag1_amp'], p['ag1_center'], p['ag1_width']) + \
         emg_burst(t, p['ag2_amp'], p['ag2_center'], p['ag2_width']) + p['tonic_agonist']
    ant = emg_burst(t, p['ant_amp'], p['ant_center'], p['ant_width']) + p['tonic_antagonist']
    return ag, ant

def simulate(tau, dt, I, B, th0):
    n = len(tau); th = np.zeros(n); om = np.zeros(n); th[0] = th0
    for i in range(n-1):
        a = (tau[i] - B*om[i])/I; om[i+1] = om[i]+a*dt; th[i+1] = th[i]+om[i]*dt
    return th, om

def compute_metrics(th, om, t, target, thresh=5.0):
    om_d = np.rad2deg(om)
    error = np.abs(np.rad2deg(th[-1]) - np.rad2deg(target))
    moving = np.abs(om_d) > thresh
    if np.any(moving):
        on = np.argmax(moving); off = len(moving)-1-np.argmax(moving[::-1]); mt = t[off]-t[on]
    else:
        on, off, mt = 0, len(t)-1, 0.0
    pk_idx = np.argmax(np.abs(om_d)); pk_v = om_d[pk_idx]
    sym = (t[pk_idx]-t[on])/mt if mt > 0 else 0.0
    overshoot = np.rad2deg(np.max(th)) - np.rad2deg(th[-1])
    return {'endpoint_error': error, 'movement_time_ms': mt*1000, 'peak_velocity': pk_v,
            'symmetry_ratio': sym, 'overshoot': overshoot}

BASE_PARAMS = {
    'normal': {'ag1_amp': 1.3, 'ag1_center': 0.10, 'ag1_width': 0.04,
               'ant_amp': 0.85, 'ant_center': 0.20, 'ant_width': 0.04,
               'ag2_amp': 0.26, 'ag2_center': 0.30, 'ag2_width': 0.03,
               'tonic_agonist': 0.03, 'tonic_antagonist': 0.03},
    'slow':   {'ag1_amp': 0.9, 'ag1_center': 0.15, 'ag1_width': 0.06,
               'ant_amp': 0.59, 'ant_center': 0.32, 'ant_width': 0.06,
               'ag2_amp': 0.14, 'ag2_center': 0.45, 'ag2_width': 0.04,
               'tonic_agonist': 0.03, 'tonic_antagonist': 0.03},
    'fast':   {'ag1_amp': 3.5, 'ag1_center': 0.06, 'ag1_width': 0.025,
               'ant_amp': 2.80, 'ant_center': 0.12, 'ant_width': 0.025,
               'ag2_amp': 0.53, 'ag2_center': 0.18, 'ag2_width': 0.02,
               'tonic_agonist': 0.03, 'tonic_antagonist': 0.03},
}
print("Simulation core loaded.")

---
## 🟢 Part 1: Remember — Building the Dataset

### Exercise 1.1: Generate Multi-Trial Dataset
Complete the `generate_experiment` function to add trial noise and collect results.

In [ ]:
# Solution 1.1: Generate multi-trial dataset
def generate_experiment(n_subjects=10, n_trials=30, conditions=('slow','normal','fast'),
                        noise_level=0.08, ant_timing_noise=0.015, seed=42):
    rng = np.random.RandomState(seed)
    rows, ts_rows = [], []
    for subj in range(n_subjects):
        subj_bias = rng.normal(0, 0.05)
        for cond in conditions:
            bp = BASE_PARAMS[cond].copy()
            for trial in range(n_trials):
                p = bp.copy()
                # Add noise: multiply each amplitude by (1 + random normal with std=noise_level)
                ### YOUR CODE HERE ### p['ag1_amp'] *= ...
                ### YOUR CODE HERE ### p['ant_amp'] *= ...
                ### YOUR CODE HERE ### p['ag2_amp'] *= ...
                p['ant_center'] += rng.normal(0, ant_timing_noise) + subj_bias * 0.01
                p['ag1_center'] += rng.normal(0, 0.005)
                for k in ['ag1_amp','ant_amp','ag2_amp']: p[k] = max(0.05, p[k])
                ag_env, ant_env = gen_envelopes(time_vec, p)
                tau = GAIN * ag_env - GAIN * ant_env
                th, om = simulate(tau, dt, I_inertia, B_damp, theta_A)
                met = compute_metrics(th, om, time_vec, theta_B)
                # Append dict: subject_id, condition, trial, all metrics, ag1_amplitude, ant_amplitude, ant_onset_ms
                ### YOUR CODE HERE ###
                if trial % 10 == 0 and subj < 3:
                    for idx in range(0, len(time_vec), 5):
                        ts_rows.append({'subject_id': f'S{subj+1:02d}', 'condition': cond,
                            'trial': trial+1, 'time_ms': time_vec[idx]*1000,
                            'theta_deg': np.rad2deg(th[idx]), 'omega_deg_s': np.rad2deg(om[idx]),
                            'emg_agonist': ag_env[idx], 'emg_antagonist': ant_env[idx]})
    return pd.DataFrame(rows), pd.DataFrame(ts_rows)

df, df_ts = generate_experiment()
print(f"Summary: {df.shape}, Time-series: {df_ts.shape}")
print(f"Subjects: {df['subject_id'].nunique()}, Conditions: {list(df['condition'].unique())}")
print(df.head().to_string())
assert df.shape == (900, 11)
print("\n\u2705 Exercise 1.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- Amplitude noise: `p['ag1_amp'] *= (1 + rng.normal(0, noise_level))`
- Dict: `rows.append({'subject_id': f'S{subj+1:02d}', 'condition': cond, 'trial': trial+1, **met, 'ag1_amplitude': p['ag1_amp'], ...})`
- Return: `return pd.DataFrame(rows), pd.DataFrame(ts_rows)`
</details>

### Exercise 1.2: Inspect the DataFrame

In [ ]:
# Solution 1.2: Inspect the DataFrame
print("Data types:\n", df.dtypes, "\n")
print("Summary statistics:\n", df.describe().round(2).to_string(), "\n")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Conditions: {df['condition'].value_counts().to_dict()}")
print("\n\u2705 Exercise 1.2 passed!")

---
## 🟡 Part 2: Understand — GroupBy and Aggregation

### Exercise 2.1: Condition-Level Summary

In [ ]:
# Exercise 2.1: Condition-level summary
# Use df.groupby('condition').agg(...) with named aggregations
cond_summary = ### YOUR CODE HERE ###

print("Condition-Level Summary\n" + "="*70)
print(cond_summary.to_string())
assert len(cond_summary) == 3
assert cond_summary.loc['fast', 'mean_peak_v'] > cond_summary.loc['slow', 'mean_peak_v']
print("\n\u2705 Exercise 2.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`df.groupby('condition').agg(mean_peak_v=('peak_velocity','mean'), std_peak_v=('peak_velocity','std'), ...)`
</details>

### Exercise 2.2: Subject × Condition Summary

In [ ]:
# Exercise 2.2: Subject x Condition summary with CV
# Group by ['subject_id', 'condition'] and compute mean, CV for peak_velocity
subj_cond = ### YOUR CODE HERE ###

print("Subject x Condition (first 9):")
print(subj_cond.head(9).to_string())

# Find most variable subject using .idxmax() on CV series
cv_by_subj = df.groupby('subject_id')['peak_velocity'].agg(lambda x: x.std()/x.mean())
most_variable = ### YOUR CODE HERE ###
print(f"\nMost variable: {most_variable} (CV={cv_by_subj[most_variable]:.3f})")
print("\n\u2705 Exercise 2.2 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- GroupBy: `df.groupby(['subject_id','condition']).agg(...)`
- CV lambda: `('peak_velocity', lambda x: x.std()/x.mean())`
- Most variable: `cv_by_subj.idxmax()`
</details>

### Exercise 2.3: Pivot Table

In [ ]:
# Exercise 2.3: Pivot table (rows=subjects, cols=conditions, values=mean peak velocity)
pivot = ### YOUR CODE HERE ###
print("Pivot: Mean Peak Velocity\n", pivot.to_string())
print("\n\u2705 Exercise 2.3 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`pd.pivot_table(df, values='peak_velocity', index='subject_id', columns='condition', aggfunc='mean').round(1)`
</details>

---
## 🟠 Part 3: Apply — Seaborn Visualization

### Exercise 3.1: Violin + Strip Plots

In [ ]:
# Exercise 3.1: Violin + strip plots for 4 metrics
fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
for ax, metric, yl in zip(axes,
    ['peak_velocity','movement_time_ms','endpoint_error','symmetry_ratio'],
    ['Peak Velocity (\u00b0/s)','Movement Time (ms)','Endpoint Error (\u00b0)','Symmetry Ratio']):
    # Violin plot: sns.violinplot(data=df, x='condition', y=metric, ...)
    ### YOUR CODE HERE ###
    # Strip plot overlay: sns.stripplot(...)
    ### YOUR CODE HERE ###
    ax.set_xlabel(''); ax.set_ylabel(yl, fontsize=10)
fig.suptitle('Movement Quality by Speed Condition', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
sns.violinplot(data=df, x='condition', y=metric, order=['slow','normal','fast'], inner=None, alpha=0.3, ax=ax)
sns.stripplot(data=df, x='condition', y=metric, order=['slow','normal','fast'], size=1.5, alpha=0.3, ax=ax, jitter=True)
```
</details>

### Exercise 3.2: Feature Pair Plot

In [ ]:
# Exercise 3.2: Pair plot for Subject S01
pair_cols = ['condition','peak_velocity','movement_time_ms','endpoint_error','symmetry_ratio']
# sns.pairplot(df[filter for S01][pair_cols], hue='condition', ...)
### YOUR CODE HERE ###
plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`sns.pairplot(df[df['subject_id']=='S01'][pair_cols], hue='condition', hue_order=['slow','normal','fast'], height=2)`
</details>

### Exercise 3.3: Correlation Heatmap

In [ ]:
# Exercise 3.3: Correlation heatmap
feat_cols = ['endpoint_error','movement_time_ms','peak_velocity','symmetry_ratio',
             'overshoot','ag1_amplitude','ant_amplitude','ant_onset_ms']
corr = ### YOUR CODE HERE ###  # df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
### YOUR CODE HERE ###  # sns.heatmap(corr, annot=True, ...)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`corr = df[feat_cols].corr()`
`sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)`
</details>

---
## 🟦 Part 4: Analyze — Trial-Averaged Kinematics

### Exercise 4.1: Velocity Profiles with Confidence Bands

In [ ]:
# Exercise 4.1: Trial-averaged velocity with confidence bands
fig, ax = plt.subplots(figsize=(8, 4.5))
# sns.lineplot(data=df_ts, x='time_ms', y='omega_deg_s', hue='condition', errorbar='se')
### YOUR CODE HERE ###
ax.axhline(0, color='gray', alpha=0.3); ax.set_xlim(0, 600)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Angular Velocity (\u00b0/s)')
ax.set_title('Trial-Averaged Velocity Profiles', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`sns.lineplot(data=df_ts, x='time_ms', y='omega_deg_s', hue='condition', hue_order=['slow','normal','fast'], ax=ax, errorbar='se')`
</details>

### Exercise 4.2: Per-Subject FacetGrid

In [ ]:
# Exercise 4.2: Per-subject FacetGrid (normal condition)
subset = df_ts[df_ts['condition']=='normal'].copy()
# g = sns.FacetGrid(subset, col='subject_id', col_wrap=3, ...)
# g.map_dataframe(sns.lineplot, x='time_ms', y='omega_deg_s', ...)
### YOUR CODE HERE ###
plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
g = sns.FacetGrid(subset, col='subject_id', col_wrap=3, height=2.5, aspect=1.3)
g.map_dataframe(sns.lineplot, x='time_ms', y='omega_deg_s', hue='trial', palette='Blues', legend=False)
g.set_xlabels('Time (ms)'); g.set_ylabels('Velocity')
```
</details>

---
## 🟣 Part 5: Evaluate — Healthy vs. Impaired

### Exercise 5.1: Generate Impaired Dataset
This code is complete — run it to create the impaired group.

In [ ]:
# Solution 5.1: Generate impaired dataset
def generate_impaired(n_subjects=5, seed=99):
    rng = np.random.RandomState(seed)
    rows = []
    for subj in range(n_subjects):
        for cond in ('slow','normal','fast'):
            bp = BASE_PARAMS[cond].copy()
            for trial in range(30):
                p = bp.copy()
                p['ag1_amp'] *= (1 + rng.normal(0, 0.08))
                p['ant_amp'] *= (1 + rng.normal(0, 0.08))
                p['ag2_amp'] *= (1 + rng.normal(0, 0.08))
                p['ant_center'] += rng.normal(0, 0.04)  # HIGH timing noise
                p['ag1_center'] += rng.normal(0, 0.005)
                for k in ['ag1_amp','ant_amp','ag2_amp']: p[k] = max(0.05, p[k])
                ag_env, ant_env = gen_envelopes(time_vec, p)
                tau = GAIN * ag_env - GAIN * ant_env
                th, om = simulate(tau, dt, I_inertia, B_damp, theta_A)
                met = compute_metrics(th, om, time_vec, theta_B)
                rows.append({'subject_id': f'P{subj+1:02d}', 'condition': cond, 'trial': trial+1,
                    **met, 'ag1_amplitude': p['ag1_amp'], 'ant_amplitude': p['ant_amp'],
                    'ant_onset_ms': p['ant_center']*1000})
    return pd.DataFrame(rows)

df_impaired = generate_impaired()
df_combined = pd.concat([df.assign(group='Healthy'), df_impaired.assign(group='Impaired')], ignore_index=True)
print(f"Healthy: {len(df)}, Impaired: {len(df_impaired)}, Combined: {len(df_combined)}")
print("\n\u2705 Exercise 5.1 passed!")

### Exercise 5.2: Comparison Plots
This code is complete — run it and study the output.

In [ ]:
# Solution 5.2: Healthy vs Impaired box plots
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, metric, yl in zip(axes, ['overshoot','peak_velocity','endpoint_error'],
                           ['Overshoot (\u00b0)','Peak Velocity (\u00b0/s)','Endpoint Error (\u00b0)']):
    sns.boxplot(data=df_combined, x='condition', y=metric, hue='group',
                order=['slow','normal','fast'], ax=ax, fliersize=2,
                palette={'Healthy':'#3498db','Impaired':'#e74c3c'})
    ax.set_xlabel(''); ax.set_ylabel(yl, fontsize=11)
    if ax != axes[-1]: ax.get_legend().remove()
    else: ax.legend(fontsize=9, loc='upper right')
fig.suptitle('Healthy vs. Cerebellar-Impaired', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("\u2705 Exercise 5.2 passed!")

### Exercise 5.3: Group Statistics

In [ ]:
# Solution 5.3: Group summary statistics
grp = df_combined.groupby(['group','condition']).agg(
    mean_error=('endpoint_error','mean'), std_error=('endpoint_error','std'),
    mean_overshoot=('overshoot','mean'), std_overshoot=('overshoot','std'),
    mean_peak_v=('peak_velocity','mean')).round(2)
print("Group x Condition Summary\n" + "="*70)
print(grp.to_string())
print("\n\u2705 Exercise 5.3 passed!")

### Exercise 5.4: Interpretation Questions

1. Which movement quality metric shows the largest difference between groups?
2. Why is the fast condition most affected in the impaired group?
3. Which features would you select for an ML classifier to distinguish groups?

**Your Answers:**

1. _[Your answer]_
2. _[Your answer]_
3. _[Your answer]_

---
## 🔴 Part 6: Create — Publication Figure

### Exercise 6.1: Multi-Panel Manuscript Figure
Design a 6-panel figure combining all your analyses.

In [ ]:
# Exercise 6.1: Design a publication-quality multi-panel figure
# Create a 2x3 grid using plt.figure + fig.add_gridspec(2, 3)
# Panel A: Example single-trial EMG
# Panel B: Trial-averaged velocity profiles
# Panel C: Violin plot of peak velocity
# Panel D: Correlation heatmap
# Panel E: Healthy vs Impaired - endpoint error
# Panel F: Healthy vs Impaired - overshoot
### YOUR CODE HERE ###
plt.savefig('manuscript_figure.png', dpi=300, bbox_inches='tight')
plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

See the Solutions notebook for a complete example using `fig.add_gridspec(2, 3)`.
</details>

---
## 🎯 Lab Summary

| Level | Skill Learned |
|---|---|
| 🟢 Remember | DataFrame construction, `.dtypes`, `.describe()` |
| 🟡 Understand | `.groupby()`, `.agg()`, `.pivot_table()` |
| 🟠 Apply | `sns.violinplot`, `sns.pairplot`, `sns.heatmap` |
| 🟦 Analyze | `sns.lineplot` with CI, `sns.FacetGrid` |
| 🟣 Evaluate | Between-group comparisons, clinical interpretation |
| 🔴 Create | Multi-panel publication figure |